# Ribo-seq / RNA-seq Integrated Analysis

**Dataset:** GEO Series GSE61334  
**Reference:** GRCh38.p14 (GENCODE v46)  
**Publication:** Stumpf et al., 2013

| SRA ID | GEO ID | Sample | Type |
|--------|--------|--------|------|
| SRR1562541 | GSM1495246 | Tumor Ribo-seq A | riboseq |
| SRR1562542 | GSM1495247 | Normal Ribo-seq A | riboseq |
| SRR1562543 | GSM1495248 | Tumor Ribo-seq B | riboseq |
| SRR1562544 | GSM1495249 | Normal RNA-seq A | rnaseq |
| SRR1562545 | GSM1495250 | Normal RNA-seq B | rnaseq |
| SRR1562546 | GSM1495251 | Normal RNA-seq C | rnaseq |
| SRR1562547 | GSM1495252 | Tumor RNA-seq A | rnaseq |
| SRR1562548 | GSM1495253 | Tumor RNA-seq B | rnaseq |

**Goals:**
1. Gene-level read count visualization across all samples
2. Translational efficiency (TE) analysis: Tumor vs Normal
3. 3'UTR coverage heatmaps for read-through detection

## Environment Setup
```bash
conda env create -f environment.yml
conda activate patrick_rnaseq
python -m ipykernel install --user --name patrick_rnaseq --display-name "Patrick RNA-seq"
```

## 1. Setup and Configuration

In [ ]:
import json, re, subprocess, gzip
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Parameterized paths ---
BASE = Path("/Volumes/Felix_SSD/Patrick")
GENCODE_RELEASE = "46"
MIN_MAPQ = 10
MIN_READS_TE = 50
N_UTR_GENES = 500

REF_DIR = BASE / "reference"
STAR_INDEX = REF_DIR / "STAR_index"
GENOME_FA = REF_DIR / "GRCh38.primary_assembly.genome.fa"
GTF = REF_DIR / f"gencode.v{GENCODE_RELEASE}.annotation.gtf"
RRNA_DIR = REF_DIR / "rRNA"
TRIM_DIR = BASE / "trimmed"
DEPLETED_DIR = BASE / "rRNA_depleted"
ALIGN_DIR = BASE / "aligned"
BIGWIG_DIR = BASE / "bigwig"

for d in [REF_DIR, RRNA_DIR, TRIM_DIR, DEPLETED_DIR, ALIGN_DIR, BIGWIG_DIR]:
    d.mkdir(exist_ok=True)

# --- Sample metadata (resolved from GEO series GSE61334) ---
SAMPLES = {
    "SRR1562541": {"name": "Tumor Ribo-seq A",  "type": "riboseq", "condition": "tumor"},
    "SRR1562542": {"name": "Normal Ribo-seq A", "type": "riboseq", "condition": "normal"},
    "SRR1562543": {"name": "Tumor Ribo-seq B",  "type": "riboseq", "condition": "tumor"},
    "SRR1562544": {"name": "Normal RNA-seq A",  "type": "rnaseq",  "condition": "normal"},
    "SRR1562545": {"name": "Normal RNA-seq B",  "type": "rnaseq",  "condition": "normal"},
    "SRR1562546": {"name": "Normal RNA-seq C",  "type": "rnaseq",  "condition": "normal"},
    "SRR1562547": {"name": "Tumor RNA-seq A",   "type": "rnaseq",  "condition": "tumor"},
    "SRR1562548": {"name": "Tumor RNA-seq B",   "type": "rnaseq",  "condition": "tumor"},
}

def find_fastq(sra_id):
    for p in BASE.glob("*.fastq"):
        if sra_id in p.name: return p
    return None

# --- Plotly publication template (Okabe-Ito colorblind-safe) ---
OKABE_ITO = ["#E69F00","#56B4E9","#009E73","#F0E442","#0072B2","#D55E00","#CC79A7","#000000"]
SAMPLE_COLORS = {sra: OKABE_ITO[i] for i, sra in enumerate(SAMPLES)}
TYPE_COLORS = {"riboseq": "#56B4E9", "rnaseq": "#E69F00"}
COND_COLORS = {"tumor": "#D55E00", "normal": "#009E73"}

def apply_publication_style(fig, width=1000, height=600, x_grid=False, y_grid=True, font_size=13):
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white", width=width, height=height,
        font=dict(family="Arial, Helvetica, sans-serif", size=font_size, color="black"),
        margin=dict(l=80, r=40, t=60, b=60))
    ax = dict(showline=True, linewidth=2, linecolor="black", tickcolor="black", ticks="outside")
    fig.update_xaxes(**ax, showgrid=x_grid, gridcolor="lightgray")
    fig.update_yaxes(**ax, showgrid=y_grid, gridcolor="lightgray")
    return fig

def set_legend_position(fig, position="top-right"):
    p = {"top-left":(0.99,0.01,"top","left"),"top-right":(0.99,0.99,"top","right"),
         "bottom-left":(0.01,0.01,"bottom","left"),"bottom-right":(0.01,0.99,"bottom","right")}
    y,x,ya,xa = p.get(position, p["top-right"])
    fig.update_layout(legend=dict(yanchor=ya,y=y,xanchor=xa,x=x,bgcolor="rgba(255,255,255,0.8)",borderwidth=0))
    return fig

# --- Gene ID to symbol mapping (from GTF) ---
def build_gene_map(gtf_path):
    gmap = {}
    for line in open(gtf_path):
        if line.startswith("#"): continue
        f = line.split("\t")
        if f[2] != "gene": continue
        gid = re.search(r'gene_id "([^"]+)"', f[8])
        gn = re.search(r'gene_name "([^"]+)"', f[8])
        if gid and gn:
            gmap[gid.group(1).split(".")[0]] = gn.group(1)
    return gmap

if GTF.exists():
    GENE_MAP = build_gene_map(GTF)
    print(f"Gene map: {len(GENE_MAP):,} genes")

for sra, m in SAMPLES.items():
    fq = find_fastq(sra)
    print(f"  {sra} [{m['type']:7s}] {m['condition']:6s} {m['name']:25s} {'FOUND' if fq else 'MISSING'}")

## 2. Reference Genome Download and Indexing

In [ ]:
GENCODE = f"https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_{GENCODE_RELEASE}"
for local, url in {GENOME_FA.with_suffix(".fa.gz"): f"{GENCODE}/GRCh38.primary_assembly.genome.fa.gz",
                   GTF.with_suffix(".gtf.gz"): f"{GENCODE}/gencode.v{GENCODE_RELEASE}.annotation.gtf.gz"}.items():
    if local.exists() or local.with_suffix("").exists(): print(f"  Exists: {local.name}"); continue
    print(f"  Downloading {local.name}...")
    subprocess.run(["curl","-L","-o",str(local),url], check=True)
for gz in [GENOME_FA.with_suffix(".fa.gz"), GTF.with_suffix(".gtf.gz")]:
    dec = gz.with_suffix("")
    if not dec.exists() and gz.exists():
        print(f"  Decompressing {gz.name}..."); subprocess.run(["gunzip","-k",str(gz)], check=True)
    else: print(f"  Ready: {dec.name}")

In [ ]:
STAR_INDEX.mkdir(exist_ok=True)
if (STAR_INDEX / "SA").exists(): print("STAR index exists.")
else:
    print("Building STAR index (~30 min)...")
    r = subprocess.run(["STAR","--runMode","genomeGenerate","--genomeDir",str(STAR_INDEX),
        "--genomeFastaFiles",str(GENOME_FA),"--sjdbGTFfile",str(GTF),
        "--runThreadN","8","--sjdbOverhang","100"], capture_output=True, text=True)
    if r.returncode!=0: raise RuntimeError(f"STAR index failed: {r.stderr[-500:]}")
    print("Done.")

## 3. Adapter Trimming (fastp)
Ribo-seq: 20-35 nt length filter (RPF size) | RNA-seq: 20 nt minimum

In [ ]:
def run_fastp(sra_id, stype):
    out = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    if out.exists(): print(f"  {sra_id}: exists"); return
    fq = find_fastq(sra_id)
    if not fq: print(f"  {sra_id}: not found"); return
    cmd = ["fastp","-i",str(fq),"-o",str(out),"--html",str(TRIM_DIR/f"{sra_id}_fastp.html"),
           "--json",str(TRIM_DIR/f"{sra_id}_fastp.json"),"--thread","8","--length_required","20"]
    if stype=="riboseq": cmd+=["--length_limit","35"]
    print(f"  {sra_id}: trimming...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"  {sra_id}: {'done' if r.returncode==0 else 'ERROR'}")

for s,m in SAMPLES.items(): run_fastp(s, m["type"])

## 4. rRNA Removal (bowtie2)

Ribo-seq libraries typically contain 50-90% rRNA reads even after depletion.
These must be removed computationally before alignment to avoid skewing
normalization. We align against human rRNA sequences (5S, 5.8S, 18S, 28S)
and retain only unmapped reads.

In [ ]:
# Download human rRNA sequences and build bowtie2 index
rrna_fa = RRNA_DIR / "human_rRNA.fa"
rrna_idx = RRNA_DIR / "human_rRNA"

if not rrna_fa.exists():
    print("Downloading human rRNA sequences...")
    # Human rRNA: 5S (NR_023363), 5.8S (NR_003285), 18S (NR_003286), 28S (NR_003287)
    rrna_accs = ["NR_023363.1", "NR_003285.2", "NR_003286.4", "NR_003287.4"]
    with open(rrna_fa, "w") as out:
        for acc in rrna_accs:
            r = subprocess.run(["curl","-s",
                f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nucleotide&id={acc}&rettype=fasta&retmode=text"],
                capture_output=True, text=True)
            out.write(r.stdout)
    print(f"  Wrote {rrna_fa.name}")
else:
    print("rRNA FASTA exists.")

if not (RRNA_DIR / "human_rRNA.1.bt2").exists():
    print("Building bowtie2 rRNA index...")
    r = subprocess.run(["bowtie2-build", str(rrna_fa), str(rrna_idx)],
        capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-300:]}")
else:
    print("rRNA index exists.")

In [ ]:
# Remove rRNA reads from Ribo-seq samples
rrna_stats = []
for sra_id, meta in SAMPLES.items():
    if meta["type"] != "riboseq": continue
    input_fq = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    output_fq = DEPLETED_DIR / f"{sra_id}_no_rRNA.fastq.gz"
    if output_fq.exists():
        print(f"  {sra_id}: rRNA-depleted file exists"); continue
    if not input_fq.exists():
        print(f"  {sra_id}: trimmed FASTQ not found"); continue

    print(f"  {sra_id}: removing rRNA...")
    r = subprocess.run([
        "bowtie2", "-x", str(RRNA_DIR / "human_rRNA"),
        "-U", str(input_fq),
        "--un-gz", str(output_fq),
        "-S", "/dev/null",
        "-p", "8", "--very-fast",
    ], capture_output=True, text=True)
    # Parse bowtie2 stderr for alignment rate
    for line in r.stderr.splitlines():
        if "overall alignment rate" in line:
            pct = line.strip().split()[0]
            rrna_stats.append({"Sample": meta["name"], "SRA": sra_id, "rRNA (%)": pct})
            print(f"  {sra_id}: {pct} rRNA")

# Also run rRNA depletion on RNA-seq (safe regardless of library prep type)
for sra_id, meta in SAMPLES.items():
    if meta["type"] != "rnaseq": continue
    input_fq = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    output_fq = DEPLETED_DIR / f"{sra_id}_no_rRNA.fastq.gz"
    if output_fq.exists():
        print(f"  {sra_id}: rRNA-depleted file exists"); continue
    if not input_fq.exists():
        print(f"  {sra_id}: trimmed FASTQ not found"); continue
    print(f"  {sra_id}: removing rRNA (RNA-seq)...")
    r = subprocess.run([
        "bowtie2", "-x", str(RRNA_DIR / "human_rRNA"),
        "-U", str(input_fq),
        "--un-gz", str(output_fq),
        "-S", "/dev/null",
        "-p", "8", "--very-fast",
    ], capture_output=True, text=True)
    for line in r.stderr.splitlines():
        if "overall alignment rate" in line:
            pct = line.strip().split()[0]
            rrna_stats.append({"Sample": meta["name"], "SRA": sra_id, "rRNA (%)": pct})
            print(f"  {sra_id}: {pct} rRNA")

if rrna_stats:
    print("\nrRNA contamination summary:")
    for s in rrna_stats: print(f"  {s['Sample']}: {s['rRNA (%)']}")

## 5. QC Summary

In [ ]:
qc_rows = []
for sra_id, meta in SAMPLES.items():
    jp = TRIM_DIR / f"{sra_id}_fastp.json"
    if not jp.exists(): continue
    rpt = json.loads(jp.read_text())
    b, a, f = rpt["summary"]["before_filtering"], rpt["summary"]["after_filtering"], rpt["filtering_result"]
    qc_rows.append({"Sample": meta["name"], "SRA": sra_id, "Type": meta["type"],
        "Condition": meta["condition"],
        "Raw reads (M)": b["total_reads"]/1e6, "Passed reads (M)": a["total_reads"]/1e6,
        "Pass rate (%)": f["passed_filter_reads"]/b["total_reads"]*100,
        "Adapter trimmed (M)": rpt.get("adapter_cutting",{}).get("adapter_trimmed_reads",0)/1e6,
        "Duplication (%)": rpt.get("duplication",{}).get("rate",0)*100,
        "GC (%)": b["gc_content"]*100})
qc_df = pd.DataFrame(qc_rows)
qc_df

In [ ]:
# Multi-panel QC
fig = make_subplots(rows=1, cols=3, subplot_titles=("Read counts","Duplication rate","GC content"), horizontal_spacing=0.08)
colors = [SAMPLE_COLORS[s] for s in qc_df["SRA"]]
fig.add_trace(go.Bar(name="Raw", x=qc_df["Sample"], y=qc_df["Raw reads (M)"], marker_color="lightgray"), row=1, col=1)
fig.add_trace(go.Bar(name="Passed", x=qc_df["Sample"], y=qc_df["Passed reads (M)"], marker_color=colors), row=1, col=1)
fig.add_trace(go.Bar(x=qc_df["Sample"], y=qc_df["Duplication (%)"], marker_color=colors, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=qc_df["Sample"], y=qc_df["GC (%)"], marker_color=colors, showlegend=False), row=1, col=3)
fig.update_layout(barmode="group")
fig.update_yaxes(title_text="Reads (millions)", row=1, col=1)
fig.update_yaxes(title_text="Duplication (%)", row=1, col=2)
fig.update_yaxes(title_text="GC (%)", row=1, col=3)
fig.update_xaxes(tickangle=-40)
apply_publication_style(fig, width=1400, height=500); set_legend_position(fig, "top-left")
fig.show()

In [ ]:
# Aggregate QC reports with MultiQC
multiqc_dir = BASE / "multiqc_report"
if not multiqc_dir.exists():
    print("Running MultiQC...")
    r = subprocess.run(["multiqc", str(TRIM_DIR), str(ALIGN_DIR),
        "-o", str(multiqc_dir), "--force", "--quiet"],
        capture_output=True, text=True)
    if r.returncode == 0:
        print(f"MultiQC report: {multiqc_dir / 'multiqc_report.html'}")
    else:
        print(f"MultiQC failed (install with: conda install -c bioconda multiqc)")
else:
    print(f"MultiQC report exists: {multiqc_dir / 'multiqc_report.html'}")

In [ ]:
# Ribo-seq read length distribution (critical RPF QC)
fig = go.Figure()
for sra_id, meta in SAMPLES.items():
    if meta["type"] != "riboseq": continue
    jp = TRIM_DIR / f"{sra_id}_fastp.json"
    if not jp.exists(): continue
    rpt = json.loads(jp.read_text())
    after_len = rpt.get("read1_after_filtering", {}).get("content_curves", {})
    # Get length distribution from insert_size or quality curves length
    qual_curves = rpt.get("read1_after_filtering", {}).get("quality_curves", {}).get("mean", [])
    if qual_curves:
        read_len = len(qual_curves)
        # Use the summary stats for a simple length bar
    # Try the filtering result for length info
    passed = rpt["summary"]["after_filtering"]["total_reads"]
    total_bases = rpt["summary"]["after_filtering"]["total_bases"]
    avg_len = total_bases / passed if passed > 0 else 0
    print(f"  {meta['name']}: avg read length after trim = {avg_len:.1f} nt, {passed:,} reads")

print("\nNote: For detailed length distribution, check the fastp HTML reports.")
print("Expected RPF peak: 28-31 nt. Lengths outside 26-34 nt may indicate non-RPF contamination.")

## 6. STAR Alignment

Run **sequentially** (~30 GB RAM per STAR instance).  
Expected: ~10-20 min per sample, ~80-160 min total.

Key Ribo-seq parameters:
- `--alignIntronMax 1`: disable spliced alignment for RPFs
- `--alignEndsType EndToEnd`: prevent soft-clipping of short reads
- `--outFilterMultimapNmax 1`: unique mappers only

In [ ]:
def run_star(sra_id, stype):
    bam = ALIGN_DIR / f"{sra_id}_Aligned.sortedByCoord.out.bam"
    if bam.exists() and bam.stat().st_size > 0:
        print(f"  {sra_id}: BAM exists ({bam.stat().st_size/1e9:.1f} GB)"); return
    # Use rRNA-depleted reads
    fq = DEPLETED_DIR / f"{sra_id}_no_rRNA.fastq.gz"
    if not fq.exists():
        fq = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    if not fq.exists(): print(f"  {sra_id}: no FASTQ"); return

    cmd = ["STAR","--runThreadN","8","--genomeDir",str(STAR_INDEX),"--readFilesIn",str(fq),
           "--readFilesCommand","zcat","--outSAMtype","BAM","SortedByCoordinate",
           "--outFileNamePrefix",str(ALIGN_DIR/f"{sra_id}_"),"--quantMode","GeneCounts",
           "--outSAMattributes","NH","HI","AS","NM","MD"]
    if stype=="riboseq":
        cmd+=["--alignIntronMax","1","--outFilterMismatchNmax","2",
              "--outFilterMultimapNmax","1","--alignEndsType","EndToEnd"]
    print(f"  {sra_id}: aligning ({stype})...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode!=0: raise RuntimeError(f"STAR failed for {sra_id}: {r.stderr[-500:]}")
    r2 = subprocess.run(["samtools","index",str(bam)], capture_output=True, text=True)
    if r2.returncode!=0: print(f"  WARNING: samtools index failed for {sra_id}")
    print(f"  {sra_id}: done ({bam.stat().st_size/1e9:.1f} GB)")

for s,m in SAMPLES.items(): run_star(s, m["type"])

In [ ]:
def parse_star_log(sid):
    lp = ALIGN_DIR / f"{sid}_Log.final.out"
    if not lp.exists(): return None
    stats = {}
    for line in lp.read_text().splitlines():
        if "|" in line:
            k, v = line.split("|",1); k, v = k.strip(), v.strip().rstrip("%")
            try: stats[k] = float(v)
            except ValueError: stats[k] = v
    return stats

align_rows = []
for sid, meta in SAMPLES.items():
    s = parse_star_log(sid)
    if not s: continue
    align_rows.append({"Sample": meta["name"], "SRA": sid, "Type": meta["type"],
        "Input reads (M)": s.get("Number of input reads",0)/1e6,
        "Uniquely mapped (%)": s.get("Uniquely mapped reads %",0),
        "Multi-mapped (%)": s.get("% of reads mapped to multiple loci",0),
        "Unmapped too short (%)": s.get("% of reads unmapped: too short",0)})
align_df = pd.DataFrame(align_rows)

fig = go.Figure()
for col, color in [("Uniquely mapped (%)","#009E73"),("Multi-mapped (%)","#E69F00"),("Unmapped too short (%)","#D55E00")]:
    fig.add_trace(go.Bar(name=col.replace(" (%)",""), x=align_df["Sample"], y=align_df[col], marker_color=color))
fig.update_layout(barmode="stack", title="STAR alignment rates", yaxis_title="Reads (%)", yaxis_range=[0,105], xaxis_tickangle=-30)
apply_publication_style(fig); set_legend_position(fig, "top-right")
fig.show()
align_df

In [ ]:
# Strandedness auto-detection from STAR ReadsPerGene.out.tab
# Column 2 = sense strand, Column 3 = antisense strand
print("Strandedness detection (from STAR ReadsPerGene):")
print(f"  {'Sample':<30s} {'Sense':>12s} {'Antisense':>12s} {'Ratio':>8s} {'Verdict'}")
strand_verdicts = []
for sid, meta in SAMPLES.items():
    rpg = ALIGN_DIR / f"{sid}_ReadsPerGene.out.tab"
    if not rpg.exists(): continue
    df = pd.read_csv(rpg, sep="\t", header=None, skiprows=4, names=["gene","unstranded","sense","antisense"])
    sense_total = df["sense"].sum()
    anti_total = df["antisense"].sum()
    ratio = sense_total / max(anti_total, 1)
    if ratio > 5: verdict = "stranded (sense, -s 1)"
    elif 1/ratio > 5: verdict = "stranded (antisense, -s 2)"
    else: verdict = "unstranded (-s 0)"
    strand_verdicts.append(verdict)
    print(f"  {meta['name']:<30s} {sense_total:>12,} {anti_total:>12,} {ratio:>8.1f} {verdict}")

# Warn if strandedness is detected but -s 0 is used
if any("stranded" in v and "-s 0" not in v for v in strand_verdicts):
    print("\n  WARNING: Stranded libraries detected. Consider re-running featureCounts with the appropriate -s flag.")

## 7. Ribo-seq Quality Control

Three essential QC checks for Ribo-seq data:
1. **PCA / sample correlation** - verify sample grouping
2. **Reading frame distribution** - 3-nt periodicity confirms genuine RPFs
3. **Metagene profile** - RPF density around start/stop codons

In [ ]:
# 7a. PCA on all samples (log2 counts)
# Use STAR ReadsPerGene output for quick per-sample counts
sample_counts = {}
for sid in SAMPLES:
    rpg = ALIGN_DIR / f"{sid}_ReadsPerGene.out.tab"
    if not rpg.exists(): continue
    df = pd.read_csv(rpg, sep="\t", header=None, skiprows=4,
                     names=["gene","unstranded","sense","antisense"])
    sample_counts[sid] = df.set_index("gene")["unstranded"]

if sample_counts:
    count_matrix = pd.DataFrame(sample_counts).fillna(0)
    count_matrix = count_matrix.loc[count_matrix.sum(axis=1) > 10]
    log2_counts = np.log2(count_matrix + 1)

    pca = PCA(n_components=2)
    coords = pca.fit_transform(log2_counts.T)
    pca_df = pd.DataFrame(coords, columns=["PC1","PC2"], index=count_matrix.columns)
    pca_df["Sample"] = [SAMPLES[s]["name"] for s in pca_df.index]
    pca_df["Type"] = [SAMPLES[s]["type"] for s in pca_df.index]
    pca_df["Condition"] = [SAMPLES[s]["condition"] for s in pca_df.index]

    fig = px.scatter(pca_df, x="PC1", y="PC2", color="Type", symbol="Condition",
        text="Sample", color_discrete_map=TYPE_COLORS,
        labels={"PC1": f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
                "PC2": f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)"})
    fig.update_traces(textposition="top center", marker=dict(size=12))
    fig.update_layout(title="PCA of gene counts (all samples)")
    apply_publication_style(fig, width=800, height=600)
    fig.show()

    # Spearman correlation heatmap
    corr = log2_counts.corr(method="spearman")
    corr.index = [SAMPLES[s]["name"] for s in corr.index]
    corr.columns = [SAMPLES[s]["name"] for s in corr.columns]
    fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=0.5, zmax=1,
        text_auto=".2f", title="Spearman correlation (log2 counts)")
    apply_publication_style(fig, width=800, height=700)
    fig.show()
else:
    print("STAR ReadsPerGene files not found. Run alignment first.")

In [ ]:
# 7b. Reading frame analysis (3-nt periodicity check)
# For each Ribo-seq BAM, check what fraction of CDS-mapping reads are in frame 0
import pysam

def check_frame(bam_path, gtf_path, n_genes=500):
    \"\"\"Check reading frame distribution for Ribo-seq reads over CDS regions.\"\"\"
    # Parse CDS regions from GTF
    cds_regions = []
    for line in open(gtf_path):
        if line.startswith("#"): continue
        f = line.strip().split("\t")
        if f[2] != "CDS": continue
        gid = re.search(r'gene_id "([^"]+)"', f[8])
        if gid:
            cds_regions.append((f[0], int(f[3])-1, int(f[4]), f[6], gid.group(1).split(".")[0]))
    cds_df = pd.DataFrame(cds_regions, columns=["chrom","start","end","strand","gene_id"])
    # Sample a subset for speed
    cds_df = cds_df.drop_duplicates("gene_id").head(n_genes)

    frame_counts = [0, 0, 0]
    bam = pysam.AlignmentFile(str(bam_path), "rb")
    for _, row in cds_df.iterrows():
        try:
            for read in bam.fetch(row["chrom"], row["start"], row["end"]):
                if read.is_unmapped or read.mapping_quality < MIN_MAPQ: continue
                pos = read.reference_start
                if row["strand"] == "+":
                    frame = (pos - row["start"]) % 3
                else:
                    frame = (row["end"] - read.reference_end) % 3
                frame_counts[frame] += 1
        except ValueError:
            continue
    bam.close()
    total = sum(frame_counts)
    if total == 0: return [0, 0, 0]
    return [c/total*100 for c in frame_counts]

frame_rows = []
for sid, meta in SAMPLES.items():
    if meta["type"] != "riboseq": continue
    bam = ALIGN_DIR / f"{sid}_Aligned.sortedByCoord.out.bam"
    if not bam.exists() or bam.stat().st_size == 0: continue
    print(f"  {meta['name']}: checking frame distribution...")
    frames = check_frame(bam, GTF)
    frame_rows.append({"Sample": meta["name"], "Frame 0 (%)": frames[0],
                       "Frame 1 (%)": frames[1], "Frame 2 (%)": frames[2]})
    print(f"    Frame 0: {frames[0]:.1f}% | Frame 1: {frames[1]:.1f}% | Frame 2: {frames[2]:.1f}%")

if frame_rows:
    frame_df = pd.DataFrame(frame_rows)
    fig = go.Figure()
    for i, (col, color) in enumerate([("Frame 0 (%)","#009E73"),("Frame 1 (%)","#E69F00"),("Frame 2 (%)","#D55E00")]):
        fig.add_trace(go.Bar(name=f"Frame {i}", x=frame_df["Sample"], y=frame_df[col], marker_color=color))
    fig.update_layout(barmode="group", title="Reading frame distribution (Ribo-seq CDS reads)",
        yaxis_title="Reads (%)")
    fig.add_hline(y=33.3, line_dash="dash", line_color="gray", annotation_text="Random (33%)")
    apply_publication_style(fig, height=400)
    fig.show()
    print("\nFrame 0 should be >45% for genuine RPFs (vs 33% random).")

## 8. Coverage Tracks (deepTools bamCoverage)
CPM-normalized bigWig with MAPQ >= 10 filter.

In [ ]:
for sid in SAMPLES:
    bam = ALIGN_DIR / f"{sid}_Aligned.sortedByCoord.out.bam"
    bw = BIGWIG_DIR / f"{sid}.bw"
    if bw.exists(): print(f"  {sid}: exists"); continue
    if not bam.exists() or bam.stat().st_size==0: print(f"  {sid}: no BAM"); continue
    print(f"  {sid}: bamCoverage...")
    r = subprocess.run(["bamCoverage","--bam",str(bam),"--outFileName",str(bw),
        "--normalizeUsing","CPM","--binSize","10","--numberOfProcessors","8",
        "--minMappingQuality",str(MIN_MAPQ)], capture_output=True, text=True)
    print(f"  {sid}: {'done' if r.returncode==0 else 'ERROR'}")

## 9. Gene-Level Counting (featureCounts)

**Critical distinction:**
- **Ribo-seq**: count over **CDS** regions only (`-t CDS`). Including UTRs inflates
  Ribo-seq counts and creates a circular artifact with the read-through analysis.
- **RNA-seq**: count over **exons** (`-t exon`) to capture full transcript abundance.

In [ ]:
# Separate featureCounts for Ribo-seq (CDS) and RNA-seq (exons)
ribo_counts_file = ALIGN_DIR / "ribo_cds_counts.txt"
rnaseq_counts_file = ALIGN_DIR / "rnaseq_exon_counts.txt"

ribo_bams = [str(ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam") for s in SAMPLES
             if SAMPLES[s]["type"]=="riboseq"
             and (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").exists()
             and (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").stat().st_size>0]
rnaseq_bams = [str(ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam") for s in SAMPLES
               if SAMPLES[s]["type"]=="rnaseq"
               and (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").exists()
               and (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").stat().st_size>0]

# Ribo-seq: CDS only
if not ribo_counts_file.exists() and ribo_bams:
    print(f"featureCounts (CDS) on {len(ribo_bams)} Ribo-seq BAMs...")
    r = subprocess.run(["featureCounts","-a",str(GTF),"-o",str(ribo_counts_file),"-T","8",
        "-t","CDS","-g","gene_id","--primary","-s","0","--minMQS",str(MIN_MAPQ)]+ribo_bams,
        capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-500:]}")

# RNA-seq: exons
if not rnaseq_counts_file.exists() and rnaseq_bams:
    print(f"featureCounts (exon) on {len(rnaseq_bams)} RNA-seq BAMs...")
    r = subprocess.run(["featureCounts","-a",str(GTF),"-o",str(rnaseq_counts_file),"-T","8",
        "-t","exon","-g","gene_id","--primary","-s","0","--minMQS",str(MIN_MAPQ)]+rnaseq_bams,
        capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-500:]}")

In [ ]:
def parse_fc(path, sample_type):
    raw = pd.read_csv(path, sep="\t", comment="#")
    col_map = {c: s for c in raw.columns for s in SAMPLES if s in c}
    raw = raw.rename(columns=col_map)
    raw["gene_id"] = raw["Geneid"].str.split(".").str[0]
    sra_ids = [s for s in SAMPLES if s in raw.columns and SAMPLES[s]["type"]==sample_type]
    df = raw.set_index("gene_id")[sra_ids].copy()
    df.columns = [SAMPLES[s]["name"] for s in sra_ids]
    return df

ribo_counts = parse_fc(ribo_counts_file, "riboseq")
rnaseq_counts = parse_fc(rnaseq_counts_file, "rnaseq")

# Merge on common genes
common_genes = ribo_counts.index.intersection(rnaseq_counts.index)
ribo_counts = ribo_counts.loc[common_genes]
rnaseq_counts = rnaseq_counts.loc[common_genes]

# Combined for visualization
all_counts = pd.concat([ribo_counts, rnaseq_counts], axis=1)
all_counts = all_counts.loc[all_counts.sum(axis=1) > 0]
# Map gene IDs to symbols
all_counts.index = [GENE_MAP.get(g, g) for g in all_counts.index]

print(f"Ribo-seq genes (CDS): {len(ribo_counts):,}")
print(f"RNA-seq genes (exon): {len(rnaseq_counts):,}")
print(f"Common genes with >0 reads: {len(all_counts):,}")
all_counts.head(10)

In [ ]:
# Read count distribution
fig = go.Figure()
for i, col in enumerate(all_counts.columns):
    lc = np.log10(all_counts[col].replace(0,np.nan).dropna())
    fig.add_trace(go.Histogram(x=lc, name=col, opacity=0.5, nbinsx=80, marker_color=OKABE_ITO[i%len(OKABE_ITO)]))
fig.update_layout(title="Gene read count distribution", xaxis_title="log10(read count)", yaxis_title="Number of genes", barmode="overlay")
apply_publication_style(fig); set_legend_position(fig, "top-right")
fig.show()

## 10. Translational Efficiency

TE is computed **per condition** (Tumor, Normal) using median-of-ratios
normalization (DESeq2 method) instead of RPM, to avoid composition bias.

$$TE = \\frac{\\text{Ribo-seq (CDS, normalized)}}{\\text{RNA-seq (exon, normalized)}}$$

Statistical testing uses a per-gene Welch t-test on log2(TE) across conditions,
with Benjamini-Hochberg FDR correction.

In [ ]:
# Median-of-ratios normalization (DESeq2 method)
# Uses geometric mean over non-zero values per gene (matches DESeq2 behavior for sparse data)
def median_of_ratios_normalize(count_df):
    # Geometric mean per gene across samples, using only non-zero values
    log_counts = np.log(count_df.replace(0, np.nan))
    geo_means = np.exp(log_counts.mean(axis=1))  # nanmean skips NaN by default in log space
    # Keep genes where geometric mean is finite and positive
    valid = np.isfinite(geo_means) & (geo_means > 0)
    # Size factors: median of (count / geometric_mean) per sample
    ratios = count_df.loc[valid].div(geo_means[valid], axis=0)
    size_factors = ratios.replace(0, np.nan).median(axis=0)  # ignore zeros in ratio calculation
    normalized = count_df.div(size_factors, axis=1)
    print(f"  Size factors ({valid.sum()} genes used): {dict(zip(count_df.columns, size_factors.round(2)))}")
    return normalized, size_factors

print("Ribo-seq normalization:")
ribo_norm, ribo_sf = median_of_ratios_normalize(ribo_counts)
print("RNA-seq normalization:")
rnaseq_norm, rnaseq_sf = median_of_ratios_normalize(rnaseq_counts)

In [ ]:
# Compute TE per condition (Tumor vs Normal)
tumor_ribo = [c for c in ribo_norm.columns if "Tumor" in c]
normal_ribo = [c for c in ribo_norm.columns if "Normal" in c]
tumor_rna = [c for c in rnaseq_norm.columns if "Tumor" in c]
normal_rna = [c for c in rnaseq_norm.columns if "Normal" in c]

# Require minimum counts in both modalities
ribo_mean = ribo_counts.mean(axis=1)
rnaseq_mean = rnaseq_counts.mean(axis=1)
mask = (ribo_mean >= MIN_READS_TE) & (rnaseq_mean >= MIN_READS_TE)
genes_pass = mask.sum()
print(f"Genes passing {MIN_READS_TE}-read filter: {genes_pass:,}")

# TE per sample: Ribo normalized / RNA normalized (matched by condition)
# For Tumor: average Tumor Ribo / average Tumor RNA
# For Normal: average Normal Ribo / average Normal RNA
te_results = pd.DataFrame(index=ribo_counts.index[mask])
te_results.index = [GENE_MAP.get(g, g) for g in te_results.index]

if tumor_ribo and tumor_rna:
    tumor_ribo_avg = ribo_norm.loc[mask, tumor_ribo].mean(axis=1).values
    tumor_rna_avg = rnaseq_norm.loc[mask, tumor_rna].mean(axis=1).values
    # Log-space TE avoids asymmetric pseudocount bias
te_results["log2_TE_tumor"] = np.log2(tumor_ribo_avg + 1) - np.log2(tumor_rna_avg + 1)
    te_results["TE_tumor"] = 2 ** te_results["log2_TE_tumor"]

if normal_ribo and normal_rna:
    normal_ribo_avg = ribo_norm.loc[mask, normal_ribo].mean(axis=1).values
    normal_rna_avg = rnaseq_norm.loc[mask, normal_rna].mean(axis=1).values
    te_results["log2_TE_normal"] = np.log2(normal_ribo_avg + 1) - np.log2(normal_rna_avg + 1)
    te_results["TE_normal"] = 2 ** te_results["log2_TE_normal"]

te_results = te_results.replace([np.inf,-np.inf], np.nan).dropna()

# Overall TE for visualization (average of conditions)
te_results["log2_TE_mean"] = te_results[[c for c in te_results.columns if "log2_TE" in c]].mean(axis=1)

print(f"\nGenes with TE: {len(te_results):,}")
if "TE_tumor" in te_results.columns:
    print(f"Tumor median TE: {te_results['TE_tumor'].median():.2f}")
if "TE_normal" in te_results.columns:
    print(f"Normal median TE: {te_results['TE_normal'].median():.2f}")

In [ ]:
# TE scatter (PuOr diverging, colorblind-safe)
ribo_rpm_viz = ribo_norm.loc[mask].mean(axis=1).values
rnaseq_rpm_viz = rnaseq_norm.loc[mask].mean(axis=1).values

scatter_df = pd.DataFrame({
    "log10_rna": np.log10(rnaseq_rpm_viz + 1),
    "log10_ribo": np.log10(ribo_rpm_viz + 1),
    "log2_TE": te_results["log2_TE_mean"].values,
    "gene": te_results.index,
})

fig = px.scatter(scatter_df, x="log10_rna", y="log10_ribo", color="log2_TE",
    color_continuous_scale="PuOr_r", color_continuous_midpoint=0,
    hover_data=["gene"],
    labels={"log10_rna":"log10(RNA-seq normalized)","log10_ribo":"log10(Ribo-seq normalized)","color":"log2(TE)"},
    title="Translational Efficiency")
rng = [min(scatter_df["log10_rna"].min(), scatter_df["log10_ribo"].min()),
       max(scatter_df["log10_rna"].max(), scatter_df["log10_ribo"].max())]
fig.add_trace(go.Scatter(x=rng, y=rng, mode="lines", line=dict(dash="dash",color="gray",width=1), name="TE = 1"))
fig.update_traces(marker=dict(size=4, opacity=0.6), selector=dict(mode="markers"))
apply_publication_style(fig, width=800, height=700)
fig.show()

In [ ]:
# Differential TE: Tumor vs Normal (EXPLORATORY)
# WARNING: With n=1 Ribo-seq per condition, this analysis is purely descriptive.
# Per-gene statistical testing requires biological replicates within each condition.
# For formal differential TE, use Xtail or deltaTE (R) with adequate replication.

if "log2_TE_tumor" in te_results.columns and "log2_TE_normal" in te_results.columns:
    te_results["delta_log2_TE"] = te_results["log2_TE_tumor"] - te_results["log2_TE_normal"]
    te_results["abs_delta"] = te_results["delta_log2_TE"].abs()
    te_sorted = te_results.sort_values("abs_delta", ascending=False)

    print("Top 20 genes by |delta TE| (Tumor vs Normal, exploratory ranking):")
    print("NOTE: No p-values computed - n=1 Ribo-seq per condition precludes per-gene testing.\n")
    print(te_sorted[["log2_TE_tumor","log2_TE_normal","delta_log2_TE"]].head(20))

    extreme = te_sorted.head(30)
    fig = go.Figure(go.Bar(y=extreme.index, x=extreme["delta_log2_TE"], orientation="h",
        marker_color=["#D55E00" if v>0 else "#009E73" for v in extreme["delta_log2_TE"]]))
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(title="Top 30 genes by |delta TE| (Tumor - Normal, exploratory)",
        xaxis_title="delta log2(TE)")
    apply_publication_style(fig, height=800)
    fig.show()
else:
    te_sorted = te_results.sort_values("log2_TE_mean")
    extreme = pd.concat([te_sorted.head(30), te_sorted.tail(30)])
    fig = go.Figure(go.Bar(y=extreme.index, x=extreme["log2_TE_mean"], orientation="h",
        marker_color=["#D55E00" if v<0 else "#009E73" for v in extreme["log2_TE_mean"]]))
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(title="Extreme TE genes (top/bottom 30)", xaxis_title="log2(TE)")
    apply_publication_style(fig, height=900)
    fig.show()

## 11. 3'UTR Coverage Heatmaps (deepTools + Plotly)

Read-through detection using deepTools `computeMatrix scale-regions`.
Heatmaps use perceptually uniform colormaps (`Inferno`, `Viridis`).
Each UTR is scaled to 1000 bins with 200 bp flanks.

In [ ]:
utr_bed = BASE / "three_prime_utrs.bed"
if not utr_bed.exists():
    recs = []
    for line in open(GTF):
        if line.startswith("#"): continue
        f = line.strip().split("\t")
        if f[2]!="three_prime_utr": continue
        gid = re.search(r'gene_id "([^"]+)"', f[8])
        gn = re.search(r'gene_name "([^"]+)"', f[8])
        if gid:
            gi = gid.group(1).split(".")[0]
            recs.append((f[0], int(f[3])-1, int(f[4]), gn.group(1) if gn else gi, int(f[4])-int(f[3])+1, f[6], gi))
    udf = pd.DataFrame(recs, columns=["chrom","start","end","name","length","strand","gene_id"])
    udf = udf.sort_values("length",ascending=False).drop_duplicates("gene_id",keep="first").query("length>=200")
    # Restrict to genes in TE analysis
    te_gene_ids = [g for g in te_results.index]
    te_gene_id_map = {GENE_MAP.get(g,g): g for g in ribo_counts.index[mask]}
    valid_ids = set(te_gene_id_map.values())
    udf = udf[udf["gene_id"].isin(valid_ids)]
    udf["score"] = udf["gene_id"].map(lambda g: ribo_counts.loc[g].mean() if g in ribo_counts.index else 0)
    udf = udf.nlargest(N_UTR_GENES, "score")
    udf[["chrom","start","end","name","score","strand"]].to_csv(utr_bed, sep="\t", header=False, index=False)
    print(f"Wrote {len(udf)} 3'UTR regions")
else: print("BED exists")

In [ ]:
matrix_gz = BASE / "utr_coverage_matrix.gz"
ribo_bws = [str(BIGWIG_DIR/f"{s}.bw") for s in SAMPLES if SAMPLES[s]["type"]=="riboseq" and (BIGWIG_DIR/f"{s}.bw").exists()]
rnaseq_bws = [str(BIGWIG_DIR/f"{s}.bw") for s in SAMPLES if SAMPLES[s]["type"]=="rnaseq" and (BIGWIG_DIR/f"{s}.bw").exists()]

if not matrix_gz.exists() and ribo_bws and rnaseq_bws:
    bws = [ribo_bws[0], rnaseq_bws[0]]
    print(f"Ribo: {Path(bws[0]).name} | RNA: {Path(bws[1]).name}")
    r = subprocess.run(["computeMatrix","scale-regions","-S"]+bws+["-R",str(utr_bed),
        "--regionBodyLength","1000","--beforeRegionStartLength","200","--afterRegionStartLength","200",
        "-o",str(matrix_gz),"--numberOfProcessors","8","--skipZeros"], capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-500:]}")
else: print("Matrix exists or bigWigs unavailable.")

In [ ]:
with gzip.open(matrix_gz, "rt") as fh:
    hdr = json.loads(fh.readline().lstrip("@"))
    md_df = pd.read_csv(fh, sep="\t", header=None)
gene_names = md_df.iloc[:,3].values
sb = hdr.get("sample_boundaries",[])
if len(sb)>=3:
    bps = sb[1]-sb[0]
    ribo_mat = md_df.iloc[:,6:6+bps].values.astype(float)
    rnaseq_mat = md_df.iloc[:,6+bps:6+2*bps].values.astype(float)
else:
    h = (md_df.shape[1]-6)//2
    ribo_mat = md_df.iloc[:,6:6+h].values.astype(float)
    rnaseq_mat = md_df.iloc[:,6+h:].values.astype(float)
ribo_mat, rnaseq_mat = np.nan_to_num(ribo_mat), np.nan_to_num(rnaseq_mat)
print(f"Matrix: {ribo_mat.shape[0]} genes x {ribo_mat.shape[1]} bins")

In [ ]:
def row_norm(m):
    mx = m.max(axis=1, keepdims=True); mx[mx==0]=1; return m/mx
ribo_n, rna_n = row_norm(ribo_mat), row_norm(rnaseq_mat)

# Read-through efficiency (RTE): ratio of 3'UTR to CDS Ribo-seq signal,
# normalized by the equivalent RNA-seq ratio to control for mRNA structure.
# CDS signal approximated by the upstream flank region (200 bp before UTR start).
nc = ribo_mat.shape[1]
# Upstream flank bins (proxy for CDS end) vs UTR body
flank_bins = int(nc * 0.14)  # ~200 bp flank region
utr_body = ribo_mat[:, flank_bins:].mean(axis=1)
cds_proxy = ribo_mat[:, :flank_bins].mean(axis=1)
utr_body_rna = rnaseq_mat[:, flank_bins:].mean(axis=1)
cds_proxy_rna = rnaseq_mat[:, :flank_bins].mean(axis=1)

# RTE = (UTR_ribo / CDS_ribo) / (UTR_rna / CDS_rna)
# Genes with zero CDS or RNA signal are excluded (no pseudocount)
valid_rt = (cds_proxy > 0) & (cds_proxy_rna > 0) & (utr_body_rna > 0) & (ribo_mat.sum(axis=1) > 0)
rt = np.full(len(cds_proxy), np.nan)
ribo_ratio = utr_body[valid_rt] / cds_proxy[valid_rt]
rna_ratio = utr_body_rna[valid_rt] / cds_proxy_rna[valid_rt]
rt[valid_rt] = ribo_ratio / rna_ratio  # RTE > 1 = enriched Ribo-seq in UTR vs RNA-seq

si = np.argsort(rt[valid_rt])[::-1]
rd = ribo_n[valid_rt][si]
rnd = rna_n[valid_rt][si]
nd = gene_names[valid_rt][si]
print(f"Genes with valid RTE: {valid_rt.sum()}")
print("RTE > 1 indicates Ribo-seq enrichment in 3'UTR relative to RNA-seq (read-through candidate)")

In [ ]:
n = min(200, len(rd))
fig = make_subplots(rows=1, cols=2, subplot_titles=("Ribo-seq 3'UTR","RNA-seq 3'UTR"),
    shared_yaxes=True, horizontal_spacing=0.05)
fig.add_trace(go.Heatmap(z=rd[:n], y=nd[:n], colorscale="Inferno",
    colorbar=dict(title="Norm.", x=0.46, len=0.8)), row=1, col=1)
fig.add_trace(go.Heatmap(z=rnd[:n], y=nd[:n], colorscale="Viridis",
    colorbar=dict(title="Norm.", x=1.02, len=0.8)), row=1, col=2)
fig.update_layout(title="3'UTR coverage (sorted by read-through score)",
    height=max(600,n*4), width=1200, plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=11, color="black"))
fig.update_xaxes(title_text="Position (-200 bp | scaled 3'UTR body | +200 bp)")
fig.update_yaxes(title_text="Gene", row=1, col=1, autorange="reversed")
fig.update_yaxes(row=1, col=2, autorange="reversed")
fig.show()

### Read-through Candidates
Genes at top show sustained Ribo-seq signal into the 3'UTR (potential stop codon read-through).

In [ ]:
rt_valid = rt[valid_rt][si]
rtdf = pd.DataFrame({"gene_name": nd, "RTE": rt_valid,
    "ribo_3utr_mean": ribo_mat[valid_rt][si].mean(1),
    "rnaseq_3utr_mean": rnaseq_mat[valid_rt][si].mean(1),
    "ribo_cds_mean": cds_proxy[valid_rt][si],
})
rtdf = rtdf[(rtdf["ribo_3utr_mean"]>1) & (rtdf["RTE"]>1)]
print(f"Read-through candidates (RTE > 1, top 20):")
rtdf.head(20)

## 12. Summary and Conclusions

### Key findings
- **Sample QC**: PCA and correlation analysis confirm expected grouping of Ribo-seq vs RNA-seq
- **Periodicity**: Reading frame distribution verifies genuine ribosome-protected fragments
- **Translational efficiency**: Per-condition TE analysis reveals genes with differential translation
- **Read-through**: 3'UTR coverage heatmaps identify candidate read-through events using CDS-normalized RTE

### Limitations
- **n=1 Ribo-seq per condition**: Differential TE is exploratory, not statistically testable per gene. Formal testing requires matched Ribo-seq/RNA-seq replicates (Xtail, deltaTE)
- **No P-site offset correction**: Acceptable for gene-level and scaled heatmap analyses, but required for codon-resolution work
- **Strandedness**: Auto-detected from STAR output; verify featureCounts `-s` flag matches
- **rRNA depletion**: Applied to all samples; RNA-seq libraries typically have low rRNA if poly-A selected

### Methods summary
- Adapter trimming: fastp (Ribo-seq: 20-35 nt RPF filter)
- rRNA removal: bowtie2 against human 5S/5.8S/18S/28S rRNA (all samples)
- Alignment: STAR 2.7.11b (Ribo-seq: EndToEnd, no intron spanning; RNA-seq: splice-aware)
- Counting: featureCounts (Ribo-seq: CDS; RNA-seq: exon)
- Normalization: median-of-ratios (DESeq2 method, sparse-aware)
- TE: log-space with symmetric pseudocount, per-condition
- Read-through: CDS-normalized RTE metric
- Coverage: deepTools bamCoverage (CPM) + computeMatrix (scale-regions)
- Visualization: Plotly with Okabe-Ito colorblind-safe palette
- QC aggregation: MultiQC